# Temperature Downscaling — Baselines

**Goal:** Evaluate baseline methods for downscaling MODIS Land Surface Temperature from 5km back to 1km.

**Pipeline:**
1. Load MODIS 1km LST as ground truth
2. Coarsen to ~5km using bicubic interpolation (simulate low-resolution input)
3. Upsample back to 1km using two baselines:
   - **BCSD** (Bias Correction Spatial Disaggregation) — classical statistical downscaling ([Wood et al., 2004](https://doi.org/10.1023/B:CLIM.0000013685.99609.9e))
   - **Lasso Regression** — L1-regularized linear model with spatial features
4. Compute metrics: RMSE, MAE, PSNR, SSIM

Reference: [Vandal et al. 2017 — DeepSD](https://arxiv.org/abs/1703.03126)

## 0. Setup

In [ ]:
!pip install -q rasterio pyhdf huggingface_hub scikit-learn scikit-image

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from collections import defaultdict

from scipy.ndimage import zoom
from sklearn.linear_model import Lasso
from skimage.metrics import structural_similarity as ssim

from pyhdf.SD import SD, SDC
from huggingface_hub import snapshot_download

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

SCALE_FACTOR = 5  # 1km * 5 = 5km

## 1. Download Data

In [ ]:
DATA_ROOT = 'data'

# Download metadata + splits + ASTER DEM
snapshot_download(
    repo_id='akhot2/downscaling',
    repo_type='dataset',
    local_dir=DATA_ROOT,
    ignore_patterns=['MODIS/*', 'Sentinel2/*', 'LULC/*'],
)

# Download MODIS subset: Jan-Mar 2022 (DOY 001-090) for all tiles
modis_patterns = [f'MODIS/*.A2022{d:03d}.*' for d in range(1, 91)]
snapshot_download(
    repo_id='akhot2/downscaling',
    repo_type='dataset',
    local_dir=DATA_ROOT,
    allow_patterns=modis_patterns,
)

MODIS_DIR = os.path.join(DATA_ROOT, 'MODIS')
ASTER_DIR = os.path.join(DATA_ROOT, 'ASTER')
print(f"MODIS files: {len(glob.glob(os.path.join(MODIS_DIR, '*.hdf')))}")
print(f"ASTER files: {len(glob.glob(os.path.join(ASTER_DIR, '*_dem.tif')))}")

## 2. Load MODIS LST Data

Read each MODIS tile's LST, convert to Celsius, and collect individual 1200x1200 tiles as ground-truth samples. Each tile is one sample — we work per-tile to avoid mosaicing complexity and keep things Colab-friendly.

In [ ]:
def read_modis_lst(hdf_path, layer='LST_Day_1km'):
    """Read MODIS LST from HDF and convert to Celsius."""
    hdf = SD(hdf_path, SDC.READ)
    sds = hdf.select(layer)
    data = sds.get().astype(np.float64)
    attrs = sds.attributes()
    scale = attrs.get('scale_factor', 0.02)
    valid_range = attrs.get('valid_range', [7500, 65535])
    data = np.where(
        (data >= valid_range[0]) & (data <= valid_range[1]),
        data * scale - 273.15,
        np.nan,
    )
    hdf.end()
    return data


def parse_modis_filename(path):
    """Extract date and tile from MODIS filename."""
    fname = os.path.basename(path)
    m = re.match(r'MOD11A1\.A(\d{7})\.(h\d{2}v\d{2})\.', fname)
    if not m:
        return None, None
    doy_str, tile = m.groups()
    year = int(doy_str[:4])
    doy = int(doy_str[4:])
    date = datetime(year, 1, 1) + timedelta(days=doy - 1)
    return date, tile

In [ ]:
modis_hdf = sorted(glob.glob(os.path.join(MODIS_DIR, 'MOD11A1.*.hdf')))
print(f"Total MODIS HDF files: {len(modis_hdf)}")

# Load all tiles, skip ones with too many NaNs (cloud cover)
MIN_VALID_FRAC = 0.5

samples = []  # list of (date, tile_id, lst_array)
for f in modis_hdf:
    date, tile = parse_modis_filename(f)
    if date is None:
        continue
    lst = read_modis_lst(f)
    valid_frac = np.sum(~np.isnan(lst)) / lst.size
    if valid_frac >= MIN_VALID_FRAC:
        samples.append((date, tile, lst))

print(f"Loaded {len(samples)} samples with >= {MIN_VALID_FRAC:.0%} valid pixels")
print(f"Tiles: {sorted(set(s[1] for s in samples))}")
print(f"Date range: {samples[0][0].strftime('%Y-%m-%d')} to {samples[-1][0].strftime('%Y-%m-%d')}")

## 3. Extract Patches & Build Train/Test Split

Extract non-overlapping patches from the 1200x1200 tiles. Each patch must be divisible by the scale factor (5), so we use 60x60 patches at HR (1km), which become 12x12 at LR (5km). We fill NaN pixels with the tile mean so the models can operate on complete arrays.

In [ ]:
PATCH_SIZE = 60  # HR patch size (must be divisible by SCALE_FACTOR)
LR_PATCH = PATCH_SIZE // SCALE_FACTOR  # 12x12
MAX_NAN_FRAC = 0.2  # skip patches with > 20% NaN

def extract_patches(lst, patch_size=PATCH_SIZE, max_nan_frac=MAX_NAN_FRAC):
    """Extract non-overlapping patches from a 2D LST array."""
    h, w = lst.shape
    patches = []
    for i in range(0, h - patch_size + 1, patch_size):
        for j in range(0, w - patch_size + 1, patch_size):
            patch = lst[i:i+patch_size, j:j+patch_size]
            nan_frac = np.sum(np.isnan(patch)) / patch.size
            if nan_frac <= max_nan_frac:
                # Fill remaining NaNs with patch mean
                patch_filled = patch.copy()
                patch_filled[np.isnan(patch_filled)] = np.nanmean(patch_filled)
                patches.append(patch_filled)
    return patches


# Extract patches from all samples
all_patches = []
for date, tile, lst in samples:
    patches = extract_patches(lst)
    all_patches.extend(patches)

all_patches = np.array(all_patches)
print(f"Total patches: {all_patches.shape[0]}, shape: {all_patches.shape[1:]}") 

# Train/test split: 80/20
np.random.seed(42)
indices = np.random.permutation(len(all_patches))
split = int(0.8 * len(indices))
train_hr = all_patches[indices[:split]]
test_hr = all_patches[indices[split:]]
print(f"Train: {len(train_hr)}, Test: {len(test_hr)}")

## 4. Coarsen HR → LR (Bicubic) & Prepare Inputs

Simulate low-resolution input by downsampling each 60x60 HR patch to 12x12 using bicubic interpolation, then upsample back to 60x60 (also bicubic) as the input for all methods.

In [ ]:
def coarsen_bicubic(hr, scale=SCALE_FACTOR):
    """Downsample HR to LR using bicubic interpolation."""
    return zoom(hr, 1.0 / scale, order=3)

def upsample_bicubic(lr, scale=SCALE_FACTOR):
    """Upsample LR to HR using bicubic interpolation."""
    return zoom(lr, scale, order=3)


# Create LR and bicubic-upsampled versions for train and test
train_lr = np.array([coarsen_bicubic(p) for p in train_hr])
test_lr = np.array([coarsen_bicubic(p) for p in test_hr])

train_bicubic = np.array([upsample_bicubic(p) for p in train_lr])
test_bicubic = np.array([upsample_bicubic(p) for p in test_lr])

print(f"HR shape: {train_hr[0].shape}, LR shape: {train_lr[0].shape}, Bicubic shape: {train_bicubic[0].shape}")

## 5. Metrics

Define evaluation metrics: RMSE, MAE, PSNR, and SSIM.

In [ ]:
def compute_metrics(pred, gt):
    """Compute RMSE, MAE, PSNR, and SSIM between prediction and ground truth."""
    mse = np.mean((pred - gt) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(pred - gt))
    # PSNR: use the data range of ground truth
    data_range = gt.max() - gt.min()
    if data_range == 0 or mse == 0:
        psnr = float('inf')
    else:
        psnr = 10 * np.log10(data_range ** 2 / mse)
    ssim_val = ssim(gt, pred, data_range=data_range)
    return {'RMSE': rmse, 'MAE': mae, 'PSNR': psnr, 'SSIM': ssim_val}


def evaluate_method(preds, gts):
    """Compute mean metrics across all test patches."""
    all_metrics = defaultdict(list)
    for pred, gt in zip(preds, gts):
        m = compute_metrics(pred, gt)
        for k, v in m.items():
            all_metrics[k].append(v)
    return {k: np.mean(v) for k, v in all_metrics.items()}

## 6. Baseline 1 — BCSD (Bias Correction Spatial Disaggregation)

BCSD ([Wood et al., 2004](https://doi.org/10.1023/B:CLIM.0000013685.99609.9e)) is a classical statistical downscaling method. For temperature (an additive variable):

1. Compute HR climatology (mean over training set at 1km)
2. Compute LR climatology (mean over training set at 5km), upsample to 1km
3. Spatial detail pattern = HR climatology − upsampled LR climatology
4. For each test sample: prediction = bicubic-upsampled LR + spatial detail pattern

This transfers fine-scale spatial structure from the climatology to each daily prediction.

In [ ]:
# Compute climatologies from training data
hr_climatology = np.mean(train_hr, axis=0)  # (60, 60)
lr_climatology = np.mean(train_lr, axis=0)  # (12, 12)
lr_clim_upsampled = upsample_bicubic(lr_climatology)  # (60, 60)

# Spatial detail pattern (additive for temperature)
spatial_detail = hr_climatology - lr_clim_upsampled

print(f"HR climatology range: [{hr_climatology.min():.1f}, {hr_climatology.max():.1f}] °C")
print(f"Spatial detail range: [{spatial_detail.min():.2f}, {spatial_detail.max():.2f}] °C")

# BCSD prediction: bicubic upsampled LR + spatial detail
test_bcsd = test_bicubic + spatial_detail[np.newaxis, :, :]

bcsd_metrics = evaluate_method(test_bcsd, test_hr)
print(f"\nBCSD metrics: {', '.join(f'{k}: {v:.4f}' for k, v in bcsd_metrics.items())}")

## 7. Baseline 2 — Lasso Regression

Per-pixel Lasso regression following [Vandal et al., 2017]. For each HR pixel, we build features from the bicubic-upsampled LR image:
- The pixel value itself
- A local 3x3 neighborhood (spatial context)
- Normalized (row, col) coordinates (positional encoding)

The Lasso (L1-regularized linear model) learns a mapping from these features to the true HR pixel value.

In [ ]:
def build_lasso_features(bicubic_patches):
    """Build per-pixel feature matrix from bicubic-upsampled patches.
    
    Features per pixel: 3x3 neighborhood (9) + normalized row/col (2) = 11 features.
    """
    n, h, w = bicubic_patches.shape
    # Pad for 3x3 neighborhood extraction
    padded = np.pad(bicubic_patches, ((0, 0), (1, 1), (1, 1)), mode='reflect')
    
    # Normalized coordinates
    rows_norm = np.linspace(0, 1, h)
    cols_norm = np.linspace(0, 1, w)
    rr, cc = np.meshgrid(rows_norm, cols_norm, indexing='ij')
    
    features = []
    for idx in range(n):
        patch_feats = np.zeros((h * w, 11))
        k = 0
        for i in range(h):
            for j in range(w):
                # 3x3 neighborhood from padded array
                neighborhood = padded[idx, i:i+3, j:j+3].ravel()  # 9 values
                patch_feats[k, :9] = neighborhood
                patch_feats[k, 9] = rr[i, j]
                patch_feats[k, 10] = cc[i, j]
                k += 1
        features.append(patch_feats)
    
    return np.vstack(features)  # (n*h*w, 11)


print("Building Lasso features (this may take a minute)...")

# Subsample training data if too large (keep Colab-friendly)
max_train = min(len(train_bicubic), 500)
X_train = build_lasso_features(train_bicubic[:max_train])
y_train = train_hr[:max_train].ravel()

print(f"Training features: {X_train.shape}, targets: {y_train.shape}")

In [ ]:
print("Training Lasso model...")
lasso = Lasso(alpha=0.01, max_iter=5000, warm_start=True)
lasso.fit(X_train, y_train)
print(f"Lasso coefficients: {lasso.coef_}")
print(f"Lasso intercept: {lasso.intercept_:.4f}")
print(f"Non-zero coefficients: {np.sum(lasso.coef_ != 0)}/{len(lasso.coef_)}")

# Predict on test set
print("\nPredicting on test set...")
X_test = build_lasso_features(test_bicubic)
y_pred = lasso.predict(X_test)
test_lasso = y_pred.reshape(test_hr.shape)

lasso_metrics = evaluate_method(test_lasso, test_hr)
print(f"\nLasso metrics: {', '.join(f'{k}: {v:.4f}' for k, v in lasso_metrics.items())}")

## 8. Results Comparison

In [ ]:
results = {
    'BCSD': bcsd_metrics,
    'Lasso': lasso_metrics,
}

# Print table
print(f"{'Method':<10} {'RMSE':>8} {'MAE':>8} {'PSNR':>8} {'SSIM':>8}")
print("-" * 46)
for method, metrics in results.items():
    print(f"{method:<10} {metrics['RMSE']:>8.4f} {metrics['MAE']:>8.4f} "
          f"{metrics['PSNR']:>8.2f} {metrics['SSIM']:>8.4f}")

In [ ]:
# Bar chart comparison
metrics_names = ['RMSE', 'MAE', 'PSNR', 'SSIM']
methods = list(results.keys())
colors = ['#2196F3', '#FF9800']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric in zip(axes, metrics_names):
    vals = [results[m][metric] for m in methods]
    bars = ax.bar(methods, vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Baseline Comparison: BCSD vs Lasso (5km → 1km)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Visual Comparison

Side-by-side comparison of a few test patches: ground truth HR, bicubic-upsampled LR input, and both baseline predictions.

In [ ]:
n_show = 4
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
col_titles = ['Ground Truth (1km)', 'Bicubic Input (5km→1km)', 'BCSD', 'Lasso']

# Pick evenly spaced test samples
show_idx = np.linspace(0, len(test_hr) - 1, n_show, dtype=int)

for row, idx in enumerate(show_idx):
    images = [test_hr[idx], test_bicubic[idx], test_bcsd[idx], test_lasso[idx]]
    vmin, vmax = test_hr[idx].min(), test_hr[idx].max()
    
    for col, (img, title) in enumerate(zip(images, col_titles)):
        ax = axes[row, col]
        im = ax.imshow(img, cmap='RdYlBu_r', vmin=vmin, vmax=vmax)
        if row == 0:
            ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])
        
        # Show per-patch RMSE for predictions
        if col >= 2:
            rmse = np.sqrt(np.mean((img - test_hr[idx]) ** 2))
            ax.text(1, 1, f'RMSE: {rmse:.2f}', transform=ax.transAxes,
                    ha='right', va='bottom', fontsize=9, color='white',
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.6))
    
    fig.colorbar(im, ax=axes[row, :].tolist(), shrink=0.8, label='LST (°C)')

plt.suptitle('Visual Comparison: Test Patches', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 10. Error Distribution

In [ ]:
# Per-pixel error distributions
bcsd_errors = (test_bcsd - test_hr).ravel()
lasso_errors = (test_lasso - test_hr).ravel()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histograms
bins = np.linspace(-5, 5, 100)
ax1.hist(bcsd_errors, bins=bins, alpha=0.6, label='BCSD', color='#2196F3', density=True)
ax1.hist(lasso_errors, bins=bins, alpha=0.6, label='Lasso', color='#FF9800', density=True)
ax1.axvline(0, color='black', linestyle='--', linewidth=0.8)
ax1.set_xlabel('Prediction Error (°C)')
ax1.set_ylabel('Density')
ax1.set_title('Error Distribution (Predicted − Ground Truth)')
ax1.legend()
ax1.grid(alpha=0.3)

# Per-patch RMSE distribution
bcsd_patch_rmse = [np.sqrt(np.mean((p - g) ** 2)) for p, g in zip(test_bcsd, test_hr)]
lasso_patch_rmse = [np.sqrt(np.mean((p - g) ** 2)) for p, g in zip(test_lasso, test_hr)]

bp = ax2.boxplot([bcsd_patch_rmse, lasso_patch_rmse], labels=['BCSD', 'Lasso'],
                  patch_artist=True)
bp['boxes'][0].set_facecolor('#2196F3')
bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#FF9800')
bp['boxes'][1].set_alpha(0.6)
ax2.set_ylabel('RMSE (°C)')
ax2.set_title('Per-Patch RMSE Distribution')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"BCSD  — median patch RMSE: {np.median(bcsd_patch_rmse):.3f} °C")
print(f"Lasso — median patch RMSE: {np.median(lasso_patch_rmse):.3f} °C")